# Master Training Pipeline: Optuna Hybrid
**Architecture:** Tri-Layer Hybrid
1. Isolation Forest (Anomaly Cleaning)
2. Prophet (Base Macroscopic Trending)
3. LightGBM (Micro Residual Corrections)
All governed by Joint Bayesian Optimization (Optuna).


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb
import optuna
import joblib
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
df = pd.read_csv('../Outputs/dataset_daily_processed.csv')
df['Date'] = pd.to_datetime(df['Date'])

features = ['Day_of_Week', 'Is_Weekend', 'Is_Holiday', 'Avg_Temp', 'Rainfall', 'Lag_1', 'Lag_7', 'Lag_30', 'Rolling_7']
target_col = 'Demand_MWh'
df = df.dropna(subset=features + [target_col]).copy()

# 2. Chronological Tri-Split (70/15/15)
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


### 3. Optuna Joint Bayesian Optimization
Instead of grid searching, we allow Optuna to trial 30 different probabilistic combinations across all three mathematical layers simultaneously.


In [ ]:
def objective(trial):
    # A. Anomaly Proxy Loop
    contamination = trial.suggest_float('contamination', 0.001, 0.05, log=True)
    temp_forest = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)
    temp_forest.fit(train_df[features])
    anomalies = temp_forest.predict(train_df[features])
    temp_train = train_df[anomalies != -1].copy()
    
    # B. Prophet Parameters
    cps = trial.suggest_float('changepoint_prior_scale', 0.01, 0.5, log=True)
    sps = trial.suggest_float('seasonality_prior_scale', 0.01, 10.0, log=True)
    
    m_base = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
                     changepoint_prior_scale=cps, seasonality_prior_scale=sps)
    
    df_p_train = temp_train[['Date', target_col]].rename(columns={'Date': 'ds', target_col: 'y'})
    m_base.fit(df_p_train)
    
    temp_train['Base_Pred'] = m_base.predict(df_p_train[['ds']])['yhat'].values
    temp_train['Residuals'] = temp_train['y'] - temp_train['Base_Pred']
    
    temp_val = val_df.copy()
    df_p_val = temp_val[['Date']].rename(columns={'Date': 'ds'})
    temp_val['Base_Pred'] = m_base.predict(df_p_val)['yhat'].values
    temp_val['Residual_Target'] = temp_val[target_col] - temp_val['Base_Pred']
    
    # C. LightGBM Parameters
    lgb_lr = trial.suggest_float('learning_rate', 0.01, 0.1, log=True)
    lgb_depth = trial.suggest_int('max_depth', 4, 10)
    lgb_leaves = trial.suggest_int('num_leaves', 20, 100)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    colsample = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    
    lgb_model = lgb.LGBMRegressor(
        learning_rate=lgb_lr, max_depth=lgb_depth, num_leaves=lgb_leaves,
        subsample=subsample, colsample_bytree=colsample,
        n_estimators=500, random_state=42
    )
    
    lgb_model.fit(
        temp_train[features], temp_train['Residuals'],
        eval_set=[(temp_val[features], temp_val['Residual_Target'])],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    lgb_val_preds = lgb_model.predict(temp_val[features])
    hybrid_preds = temp_val['Base_Pred'] + lgb_val_preds
    
    return mean_absolute_error(temp_val[target_col], hybrid_preds)

print("Starting Optuna Study...")
sampler = optuna.samplers.TPESampler(seed=0)
study = optuna.create_study(direction='minimize', sampler=sampler)
study.optimize(objective, n_trials=30, show_progress_bar=True)
best_p = study.best_params
print(f"Best Parameters: {best_p}")


### 4. Final Training & Artifact Serialization
We retrain the architecture with the champion hyperparameters found by Optuna and serialize them to `.joblib` for the Inference pipeline.


In [ ]:
# 1. Final Anomaly Clean
iso_forest = IsolationForest(n_estimators=300, contamination=best_p['contamination'], random_state=42)
iso_forest.fit(train_df[features])
train_anomalies = iso_forest.predict(train_df[features])
train_df_clean = train_df[train_anomalies != -1].copy()

# 2. Final Prophet
df_p_train = train_df_clean[['Date', target_col]].rename(columns={'Date': 'ds', target_col: 'y'})
prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
                        changepoint_prior_scale=best_p['changepoint_prior_scale'], 
                        seasonality_prior_scale=best_p['seasonality_prior_scale'])
prophet_model.fit(df_p_train)

train_df_clean['Base_Pred'] = prophet_model.predict(df_p_train[['ds']])['yhat'].values
train_df_clean['Residuals'] = train_df_clean['y'] - train_df_clean['Base_Pred']

# 3. Final LightGBM
df_p_val = val_df[['Date']].rename(columns={'Date': 'ds'})
val_df['Base_Pred'] = prophet_model.predict(df_p_val)['yhat'].values
y_val_residuals = val_df[target_col] - val_df['Base_Pred']

lgb_model = lgb.LGBMRegressor(
    learning_rate=best_p['learning_rate'], max_depth=best_p['max_depth'], num_leaves=best_p['num_leaves'],
    subsample=best_p['subsample'], colsample_bytree=best_p['colsample_bytree'],
    n_estimators=1000, random_state=42
)

lgb_model.fit(
    train_df_clean[features], train_df_clean['Residuals'],
    eval_set=[(val_df[features], y_val_residuals)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

# 4. Serialize Models for Production
os.makedirs('../Models', exist_ok=True)
joblib.dump(prophet_model, '../Models/prophet_model.joblib')
joblib.dump(lgb_model, '../Models/lgbm_model.joblib')
joblib.dump(iso_forest, '../Models/iso_forest.joblib')
print("\n!!! SUCCESS: Model weights saved to /Models folder !!!")
